In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install auto-round --break-system-packages

In [ ]:
# !pip install compressed-tensors --break-system-packages

In [1]:
import os
import shutil

import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from transformers import AutoModelForImageTextToText, AutoProcessor



W1008 12:38:07.395000 10780 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1008 12:38:07.424000 10780 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.14.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA L40S
VRAM: 44.4 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "LiquidAI/d1-3B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Reconstructing (incomplete total...): |           |  0.00B /  0.00B            

Fetching 15 files: 100%|██████████████████████| 15/15 [00:00<00:00, 517.37it/s]
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            ✓ Downloaded
  path: /workspace/local_model
Download complete:                                         |  0.00B            
Reconstruction complete: |                        |  0.00B /  0.00B            


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

In [8]:
model

Lfm2VlForConditionalGeneration(
  (model): Lfm2VlModel(
    (vision_tower): Siglip2VisionModel(
      (embeddings): Siglip2VisionEmbeddings(
        (patch_embedding): Linear(in_features=768, out_features=1152, bias=True)
        (position_embedding): Embedding(256, 1152)
      )
      (encoder): Siglip2Encoder(
        (layers): ModuleList(
          (0-26): 27 x Siglip2EncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (self_attn): Siglip2Attention(
              (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
            (mlp): Siglip2MLP(
              (activ

In [9]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 800,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 2,
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": False,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [10]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-10-08 12:38:16 WARNING autoround.py L489: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-08 12:38:16 WARNING autoround.py L489: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-10-08 12:38:16 WARNING autoround.py L489: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [11]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,llm_compressor", inplace=True
)

2026-10-08 12:38:16 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): model.vision_tower.encoder.layers.[0-26].mlp.fc1, model.vision_tower.encoder.layers.[0-26].mlp.fc2
2026-10-08 12:38:17 INFO base.py L1236: 'enable_torch_compile' is disabled. Enabling it can reduce tuning cost by about 20%.
2026-10-08 12:38:17 INFO base.py L1372: `torch.compile` is disabled, as the user disabled it
2026-10-08 12:38:17 INFO orchestrator.py L594: start to cache block inputs
2026-10-08 12:38:17 INFO mllm.py L86: Using MLLM template: lfm2_vl
2026-10-08 12:38:17 INFO mllm.py L125: Multimodal model with non-MLLM calibration dataset 'NeelNanda/pile-10k' and quant_nontext_module=False: using the standard text dataloader (vision/audio towers are not being quantized, so text-only calibration through the full-model forward is sufficient).
2026-10-08 12:38:17 INFO calib_dataset.py L1229: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-10-08

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

packing: 100%|██████████| 220/220 [00:06<00:00, 31.90it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-08 12:55:18 INFO device.py L1560: 'peak_ram': 11.23GB, 'peak_vram': 14.43GB


(Lfm2VlForConditionalGeneration(
   (model): Lfm2VlModel(
     (vision_tower): Siglip2VisionModel(
       (embeddings): Siglip2VisionEmbeddings(
         (patch_embedding): Linear(in_features=768, out_features=1152, bias=True)
         (position_embedding): Embedding(256, 1152)
       )
       (encoder): Siglip2Encoder(
         (layers): ModuleList(
           (0-26): 27 x Siglip2EncoderLayer(
             (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
             (self_attn): Siglip2Attention(
               (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
               (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
               (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
               (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
             )
             (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True, bias=True)
             (mlp): Siglip2MLP(
 

In [12]:
# Files required for remote code execution, tokenization, and processor initialization
auxiliary_files: list[str] = [
    # Custom architecture / remote code execution
    "modeling_d1.py",
    "lfm2_vl.py",
    "hybrid.py",
    "api.py",
    "runner.py",
    "prompt.py",
    "LICENSE",
]

export_subdirs: list[str] = [
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"),
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-gptq"),
    os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/llm-compressor-wint-a16"),
]

for export_dir in export_subdirs:
    if not os.path.exists(export_dir):
        continue

    print(f"\n--> Injecting inference assets into: {export_dir}")
    for filename in auxiliary_files:
        src = os.path.join(LOCAL_PATH, filename)
        dst = os.path.join(export_dir, filename)

        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"    Copied: {filename}")
        else:
            print(f"    Warning: Missing {src}")


--> Injecting inference assets into: ./AutoRound/local_model-w4g32/auto-round-auto-gptq
    Copied: modeling_d1.py
    Copied: lfm2_vl.py
    Copied: hybrid.py
    Copied: api.py
    Copied: runner.py
    Copied: prompt.py
    Copied: LICENSE

--> Injecting inference assets into: ./AutoRound/local_model-w4g32/auto-gptq
    Copied: modeling_d1.py
    Copied: lfm2_vl.py
    Copied: hybrid.py
    Copied: api.py
    Copied: runner.py
    Copied: prompt.py
    Copied: LICENSE

--> Injecting inference assets into: ./AutoRound/local_model-w4g32/llm-compressor-wint-a16
    Copied: modeling_d1.py
    Copied: lfm2_vl.py
    Copied: hybrid.py
    Copied: api.py
    Copied: runner.py
    Copied: prompt.py
    Copied: LICENSE


In [13]:
def push_to_hub(local_dir, repo_name, token):
    if not os.path.exists(local_dir):
        print(f"[Hub] Directory not found, skipping: {local_dir}")
        return

    full_repo_id = f"{HF_USER}/{repo_name}"
    try:
        api = HfApi()
        print(f"[Hub] Preparing repo: {full_repo_id}...")
        create_repo(full_repo_id, repo_type="model", private=False, token=token, exist_ok=True)
        
        print(f"[Hub] Uploading folder from {local_dir}...")
        api.upload_folder(
            folder_path=local_dir,
            repo_id=full_repo_id,
            repo_type="model",
            token=token
        )
        print(f"[Hub] Upload complete: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] Error uploading {full_repo_id}: {e}")


In [14]:

base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()


In [15]:

if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g32/auto-round-auto-gptq"), 
        f"{base_name}-W4A16-AutoRound", 
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload.")

[Hub] Preparing repo: Vishva007/d1-3B-W4A16-AutoRound...
[Hub] Uploading folder from ./AutoRound/local_model-w4g32/auto-round-auto-gptq...
[Hub] Upload complete: https://huggingface.co/Vishva007/d1-3B-W4A16-AutoRound
